## Launch and set up NVIDIA A100 or A30 server - with python-chi

At the beginning of the lease time for your bare metal server, we will bring up our GPU instance. We will use the `python-chi` Python API to Chameleon to provision our server.

We will execute the cells in this notebook inside the Chameleon Jupyter environment.

Run the following cell, and make sure the correct project is selected. Also **change the site to CHI@TACC or CHI@UC**, depending on where your reservation is.

In [1]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease
import os

context.version = "1.0" 
context.choose_project()
context.choose_site(default="CHI@UC")

Change the string in the following cell to reflect the name of *your* lease (**with your own net ID**), then run it to get your lease:

In [2]:
# runs in Chameleon Jupyter environment
l = lease.get_lease(f"serve_model_jl16053") 
l.show()

HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>serve_model_jl…

Lease Details:
Name: serve_model_jl16053
ID: eb60cbdf-1934-4804-b647-0ed57e2db8d1
Status: ACTIVE
Start Date: 2026-03-25 22:04:00
End Date: 2026-03-26 02:00:00
User ID: b7c538195f4db5e841c80a0d766eab47e526daf84670f0bb8a8cf52e0f2935cd
Project ID: 7c0a7a1952e44c94aa75cae1ff5dc9b4

Node Reservations:
ID: 8d07524a-793c-4863-9610-20b9255d78a6, Status: active, Min: 1, Max: 1

Floating IP Reservations:

Network Reservations:

Flavor Reservations:

Events:


The status should show as “ACTIVE” now that we are past the lease start time.

The rest of this notebook can be executed without any interactions from you, so at this point, you can save time by clicking on this cell, then selecting “Run” \> “Run Selected Cell and All Below” from the Jupyter menu.

As the notebook executes, monitor its progress to make sure it does not get stuck on any execution error, and also to see what it is doing!

We will use the lease to bring up a server with the `CC-Ubuntu24.04-CUDA` disk image.

> **Note**: the following cell brings up a server only if you don’t already have one with the same name! (Regardless of its error state.) If you have a server in ERROR state already, delete it first in the Horizon GUI before you run this cell.

In [3]:
# runs in Chameleon Jupyter environment
username = os.getenv('USER') # all exp resources will have this prefix
s = server.Server(
    f"node-serve-model-{username}", 
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-CUDA"
)
s.submit(idempotent=True)

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server node-serve-model-jl16053_nyu_edu's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-serve-model-jl16053_nyu_edu
Id,ceea1866-794a-4b10-8807-4374e742a759
Status,ACTIVE
Image Name,CC-Ubuntu24.04-CUDA
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.140.83.114 (v4) Type: fixed MAC: 14:23:f2:a3:f8:00
Network Name,sharednet1
Created At,2026-03-25T22:04:27Z
Keypair,trovi-8abe1a4
Reservation Id,8d07524a-793c-4863-9610-20b9255d78a6
Host Id,c15c5d0cd98629a41c320d11364f137f4320899eed52f609fb88500c


Note: security groups are not used at Chameleon bare metal sites, so we do not have to configure any security groups on this instance.

Then, we’ll associate a floating IP with the instance, so that we can access it over SSH.

In [4]:
# runs in Chameleon Jupyter environment
s.associate_floating_ip()

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


'192.5.87.187'

In [5]:
# runs in Chameleon Jupyter environment
s.refresh()
s.check_connectivity()

Checking connectivity to 192.5.87.187 port 22.


Connection successful


In the output below, make a note of the floating IP that has been assigned to your instance (in the “Addresses” row).

In [6]:
# runs in Chameleon Jupyter environment
s.refresh()
s.show(type="widget")

Attribute,node-serve-model-jl16053_nyu_edu
Id,ceea1866-794a-4b10-8807-4374e742a759
Status,ACTIVE
Image Name,CC-Ubuntu24.04-CUDA
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.140.83.114 (v4) Type: fixed MAC: 14:23:f2:a3:f8:00 IP: 192.5.87.187 (v4) Type: floating MAC: 14:23:f2:a3:f8:00
Network Name,sharednet1
Created At,2026-03-25T22:04:27Z
Keypair,trovi-8abe1a4
Reservation Id,8d07524a-793c-4863-9610-20b9255d78a6
Host Id,c15c5d0cd98629a41c320d11364f137f4320899eed52f609fb88500c


### Retrieve code and notebooks on the instance

Now, we can use `python-chi` to execute commands on the instance, to set it up. We’ll start by retrieving the code and other materials on the instance.

In [7]:
# runs in Chameleon Jupyter environment
s.execute("git clone https://github.com/teaching-on-testbeds/serve-model-chi")

/opt/conda/lib/python3.12/site-packages/paramiko/client.py:885: UserWarning: Unknown ssh-ed25519 host key for 192.5.87.187: b'1c187903baa6e9b8e44f0a7cd1a17f69'
  warnings.warn(
Cloning into 'serve-model-chi'...


<Result cmd='git clone https://github.com/teaching-on-testbeds/serve-model-chi' exited=0>

### Set up Docker

To run the serving and inference experiments in this lab, we will use Docker containers that already include the required runtime libraries. In this step, we set up Docker on the server so we can launch those containers.

In [8]:
# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

# Executing docker install script, commit: f381ee68b32e515bb4dc034b339266aff1fbc460


+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

Restarting services...
 systemctl restart cache-vendordata.service packagekit.service

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu noble stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install docker-ce docker-ce-cli containerd.io docker-compose-plugin docker-ce-rootles

  UNIT                                                                                                  LOAD   ACTIVE SUB       DESCRIPTION
  proc-sys-fs-binfmt_misc.automount                                                                     loaded active running   Arbitrary Executable File Formats File System Automount Point
  sys-bus-pci-drivers-nvidia.device                                                                     loaded active plugged   /sys/bus/pci/drivers/nvidia
  sys-devices-pci0000:78-0000:78:03.1-0000:7b:00.0-net-eno12399np0.device                               loaded active plugged   BCM57504 NetXtreme-E 10Gb/25Gb/40Gb/50Gb/100Gb/200Gb Ethernet (NetXtreme-E BCM57504 4x25G OCP3.0)
  sys-devices-pci0000:78-0000:78:03.1-0000:7b:00.0-ptp-ptp0.device                                      loaded active plugged   /sys/devices/pci0000:78/0000:78:03.1/0000:7b:00.0/ptp/ptp0
  sys-devices-pci0000:78-0000:78:03.1-0000:7b:00.1-net-eno12409np1.device                            

INFO: Docker daemon enabled and started



ded active exited    Cloud-init: Network Stage
  console-setup.service                                                                                 loaded active exited    Set console font and keymap
  containerd.service                                                                                    loaded active running   containerd container runtime
  cron.service                                                                                          loaded active running   Regular background program processing daemon
  dbus.service                                                                                          loaded active running   D-Bus System Message Bus
  docker.service                                                                                        loaded active running   Docker Application Container Engine
  finalrd.service                                                                                       loaded active exited    Create final runtime d

+ sh -c docker version


ir for shutdown pivot root
  firewalld.service                                                                                     loaded active running   firewalld - dynamic firewall daemon
  getty@tty1.service                                                                                    loaded active running   Getty on tty1
  keyboard-setup.service                                                                                loaded active exited    Set the console keyboard layout
  kmod-static-nodes.service                                                                             loaded active exited    Create List of Static Device Nodes
  ldconfig.service                                                                                      loaded active exited    Rebuild Dynamic Linker Cache
  lvm2-monitor.service                                                                                  loaded active exited    Monitoring of LVM2 mirrors, snapshots etc. using dmeventd o

<Result cmd='sudo groupadd -f docker; sudo usermod -aG docker $USER' exited=0>

### Set up the NVIDIA container toolkit

We will also install the NVIDIA container toolkit, with which we can access GPUs from inside our containers.

In [9]:
# runs in Chameleon Jupyter environment
s.execute("curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg \
  && curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | \
    sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
    sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list")
s.execute("sudo apt update")
s.execute("sudo apt-get install -y nvidia-container-toolkit")
s.execute("sudo nvidia-ctk runtime configure --runtime=docker")
# for https://github.com/NVIDIA/nvidia-container-toolkit/issues/48
s.execute("sudo jq 'if has(\"exec-opts\") then . else . + {\"exec-opts\": [\"native.cgroupdriver=cgroupfs\"]} end' /etc/docker/daemon.json | sudo tee /etc/docker/daemon.json.tmp > /dev/null && sudo mv /etc/docker/daemon.json.tmp /etc/docker/daemon.json")
s.execute("sudo systemctl restart docker")

deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://nvidia.github.io/libnvidia-container/stable/deb/$(ARCH) /
#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://nvidia.github.io/libnvidia-container/experimental/deb/$(ARCH) /


Hit:1 https://download.docker.com/linux/ubuntu noble InRelease
Get:2 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease [1477 B]
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:5 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease
Get:8 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  Packages [25.2 kB]
Fetched 26.7 kB in 1s (35.2 kB/s)
Reading package lists...
Building dependency tree...
Reading state information...
228 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libnvidia-container-tools libnvidia-container1 nvidia-contain

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 8126 kB in 0s (39.5 MB/s)
Selecting previously unselected package libnvidia-container1:amd64.
(Reading database ... 113876 files and directories currently installed.)
Preparing to unpack .../libnvidia-container1_1.19.0-1_amd64.deb ...
Unpacking libnvidia-container1:amd64 (1.19.0-1) ...
Selecting previously unselected package libnvidia-container-tools.
Preparing to unpack .../libnvidia-container-tools_1.19.0-1_amd64.deb ...
Unpacking libnvidia-container-tools (1.19.0-1) ...
Selecting previously unselected package nvidia-container-toolkit-base.
Preparing to unpack .../nvidia-container-toolkit-base_1.19.0-1_amd64.deb ...
Unpacking nvidia-container-toolkit-base (1.19.0-1) ...
Selecting previously unselected package nvidia-container-toolkit.
Preparing to unpack .../nvidia-container-toolkit_1.19.0-1_amd64.deb ...
Unpacking nvidia-container-toolkit (1.19.0-1) ...
Setting up nvidia-container-toolkit-base (1.19.0-1) ...
Created symlink /etc/systemd/system/multi-user.target.wants/nvidia-

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
time="2026-03-25T22:16:48Z" level=info msg="Config file does not exist; using empty config"
time="2026-03-25T22:16:48Z" level=info msg="Wrote updated config to /etc/docker/daemon.json"
time="2026-03-25T22:16:48Z" level=info msg="It is recommended that docker daemon be restarted."


<Result cmd='sudo systemctl restart docker' exited=0>

## Open an SSH session

Finally, open an SSH sesson on your server. From your local terminal, run

    ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D

where

-   in place of `~/.ssh/id_rsa_chameleon`, substitute the path to your own key that you had uploaded to CHI@TACC
-   in place of `A.B.C.D`, use the floating IP address you just associated to your instance.